# 01 · Eksplorasi data Vine

Melihat isi dataset sebelum training: jumlah per label, panjang kalimat, kata yang paling khas per label, dan kalimat yang bertabrakan dengan aturan kata darurat.

Jalankan di laptop atau Colab. Tidak butuh GPU.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'training'))
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 120)


## 1. Jumlah kalimat per label

In [ ]:
intent = pd.read_csv('data/intent/raw/kalimat_pasien.csv')
emosi = pd.read_csv('data/emotion/raw/kalimat_emosi.csv')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
intent['label'].value_counts().plot.barh(ax=ax[0], title='Maksud (alarm)')
emosi['label'].value_counts().plot.barh(ax=ax[1], title='Emosi')
plt.tight_layout(); plt.show()
print('Target minimal: 100 kalimat per label')


## 2. Panjang kalimat

Kalimat yang terlalu seragam panjangnya membuat model mudah 'curang'. Ucapan pasien sungguhan bervariasi.

In [ ]:
for nama, df in (('maksud', intent), ('emosi', emosi)):
    df['jumlah_kata'] = df['teks'].str.split().str.len()
    print(nama); print(df.groupby('label')['jumlah_kata'].describe()[['mean', 'min', 'max']].round(1), '\n')


## 3. Kata paling khas per label

Kalau satu kata terlalu dominan (misalnya semua kalimat darurat memakai kata 'sakit'), model akan salah menganggap keluhan ringan sebagai darurat. Tambahkan contoh di label lain yang juga memakai kata itu.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
def kata_khas(df, n=10):
    vec = TfidfVectorizer(ngram_range=(1, 1))
    X = vec.fit_transform(df['teks'])
    kata = np.array(vec.get_feature_names_out())
    for lb in df['label'].unique():
        skor = np.asarray(X[(df['label'] == lb).values].mean(axis=0)).ravel()
        print(f'{lb:>17}:', ', '.join(kata[skor.argsort()[::-1][:n]]))
kata_khas(intent); print(); kata_khas(emosi)


## 4. Tabrakan dengan aturan kata darurat

Kalimat yang labelnya bukan darurat tapi tetap ditandai merah oleh aturan. Ini akan menjadi alarm palsu di sistem nyata. Kalau terlalu banyak, persempit pola di `config/thresholds.yaml`.

In [ ]:
from src.intent.safety_rules import SafetyRules
rules = SafetyRules()
intent['aturan'] = intent['teks'].map(lambda t: rules.check(t).level)
print('Bukan darurat tapi aturan bilang merah:')
print(intent[(intent['aturan'] == 'merah') & (intent['label'] != 'darurat')][['teks', 'label']])
print('\nDarurat yang TIDAK tertangkap aturan (harus ditangkap model):', ((intent['label'] == 'darurat') & (intent['aturan'] != 'merah')).sum(), 'kalimat')


## 5. Dataset persona

In [ ]:
import json, glob
rows = [json.loads(l) for f in glob.glob('data/persona/*.jsonl') for l in open(f, encoding='utf-8') if l.strip()]
persona = pd.DataFrame(rows)
print(pd.crosstab(persona['persona'], persona['emosi']))
persona['kata_balasan'] = persona['ai'].str.split().str.len()
print('\nRata-rata panjang balasan (kata):'); print(persona.groupby('persona')['kata_balasan'].mean().round(1))
